# Step 5: Profit levers for the board

This notebook turns Steps 2–4 into board decisions with dollar figures, ranked by size. It adds one new lever, branch pricing, and an experiment design for the 10% discount.

**How to read the dollar figures.** Every dollar figure is a **conditional annualised gross-profit opportunity**: H1 gross-profit difference × 365/181, *if* the stated condition holds. They are **not** net profit (no labour, rent or overheads), **not** promised savings, and they **must not be added together** into one forecast: each depends on a different condition, and they interact (for example, Garden's price lever depends on which cost table is right).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw = pd.read_csv(ROOT / "data" / "raw" / "stories_transactions.csv")
raw["Date"] = pd.to_datetime(raw["Date"])
ANN = 365 / 181
DAYS = raw["Date"].nunique()
price = raw.groupby(["Branch", "Product"])["Unit Price (USD)"].first().unstack(0)
cost = raw.groupby(["Branch", "Product"])["Unit Cost (USD)"].first().unstack(0)
assert (raw.groupby(["Branch", "Product"])[["Unit Price (USD)", "Unit Cost (USD)"]].nunique() == 1).all().all()
print(f"{len(raw):,} lines, {DAYS} days; prices and costs are fixed per branch and product over the period")
print("Price ratio vs Harbour, all 45 products (min-max):",
      {b: f"{(price[b]/price['Harbour']).min():.3f}-{(price[b]/price['Harbour']).max():.3f}" for b in ["Garden", "Uptown"]})
print("Cost ratio vs Harbour (min-max):",
      {b: f"{(cost[b]/cost['Harbour']).min():.3f}-{(cost[b]/cost['Harbour']).max():.3f}" for b in ["Garden", "Uptown"]})

51,177 lines, 181 days; prices and costs are fixed per branch and product over the period
Price ratio vs Harbour, all 45 products (min-max): {'Garden': '0.951-0.953', 'Uptown': '0.912-0.916'}
Cost ratio vs Harbour (min-max): {'Garden': '1.426-1.439', 'Uptown': '0.952-0.965'}


## 1. Garden cost audit (largest lever; detail in Step 3)

In [2]:
g = raw[raw.Branch == "Garden"].copy()
g["harbour_cost"] = g.Product.map(cost["Harbour"])
gap = ((g["Unit Cost (USD)"] - g["harbour_cost"]) * g.Quantity).sum()
print(f"Fact: Garden recorded-cost difference vs Harbour unit costs, H1: ${gap:,.0f}")
print(f"Conditional annualised gross-profit opportunity: ${gap*ANN:,.0f}/yr "
      "(only if Garden really pays more than Harbour AND equivalent supply terms are feasible)")

Fact: Garden recorded-cost difference vs Harbour unit costs, H1: $15,923
Conditional annualised gross-profit opportunity: $32,111/yr (only if Garden really pays more than Harbour AND equivalent supply terms are feasible)


## 2. Branch pricing: what a price change must survive

**Facts.** On all 45 products, Garden prices are about 95.2% of Harbour's and Uptown prices about 91.4%. Prices never change in the data, so **the data cannot show how customers would react to a price change**. Similar hours or basket sizes across branches do not show similar willingness to pay.

**What we can calculate** is the break-even: how much volume a branch could lose before a price rise reduces gross profit. If volume falls by less than that, gross profit rises.

Break-even volume loss = 1 − (current gross profit / gross profit at new prices with unchanged volume). Discount rates are kept as recorded.

Garden is shown under **recorded costs** and under **normalised costs** (Harbour's unit costs). Its break-even depends heavily on which cost table is right, which is another reason to do the cost audit first.

In [3]:
def price_scenario(branch, step, cost_basis):
    x = raw[raw.Branch == branch]
    unit_cost = x["Unit Cost (USD)"] if cost_basis == "recorded" else x.Product.map(cost["Harbour"])
    new_price = x["Unit Price (USD)"] + step * (x.Product.map(price["Harbour"]) - x["Unit Price (USD)"])
    rev0 = x["Line Revenue (USD)"].sum()
    rev1 = (x.Quantity * new_price * (1 - x["Discount Rate"])).sum()
    gp0 = rev0 - (x.Quantity * unit_cost).sum()
    gp1 = gp0 + (rev1 - rev0)
    return {"branch": branch, "cost basis": cost_basis,
            "price change": f"{'half' if step == 0.5 else 'full'} gap to Harbour (+{rev1/rev0-1:.1%})",
            "H1 gross profit now": round(gp0), "margin now": f"{gp0/rev0:.1%}",
            "opportunity if volume unchanged ($/yr)": round((rev1 - rev0) * ANN),
            "break-even volume loss": f"{1 - gp0/gp1:.1%}"}

pricing = pd.DataFrame([price_scenario(b, s, c) for b, c in [("Uptown", "recorded"), ("Garden", "recorded"), ("Garden", "normalised (Harbour costs)")]
                        for s in [0.5, 1.0]])
pricing

,branch,cost basis,price change,H1 gross profit now,margin now,opportunity if volume unchanged ($/yr),break-even volume loss
0,Uptown,recorded,half gap to Harbour (+4.7%),34882,62.6%,5272,7.0%
1,Uptown,recorded,full gap to Harbour (+9.4%),34882,62.6%,10545,13.0%
2,Garden,recorded,half gap to Harbour (+2.5%),46794,46.9%,5026,5.1%
3,Garden,recorded,full gap to Harbour (+5.0%),46794,46.9%,10053,9.6%
4,Garden,normalised (Harbour costs),half gap to Harbour (+2.5%),62718,62.9%,5026,3.8%
5,Garden,normalised (Harbour costs),full gap to Harbour (+5.0%),62718,62.9%,10053,7.4%


**Reading.**
* **Uptown** (costs about 96% of Harbour's, so no cost question): closing the full price gap is +9.4%. It is a gross-profit opportunity of about $10.5k/yr **if volume is unchanged**, and it stays positive unless Uptown loses about 13% or more of its volume.
* **Garden**: the break-even is *higher* under recorded costs (9.6% for the full gap) than under normalised costs (7.4%). With a thin recorded margin, each lost sale costs less recorded profit, so a price rise looks safer than it may be. The opportunity in dollars is the same under both cost bases, because costs do not change with price. Because Garden's cost table is in question (Step 3) and Garden is the second-largest branch, **we recommend the pricing experiment at Uptown first**.

## 3. Uptown pricing experiment: design and stop/review rules

* **Change:** raise Uptown prices by **half the gap** to Harbour first (about +4.7%), not the full gap, to limit downside.
* **Comparison:** Uptown's daily purchases and gross profit relative to Garden + Harbour on the same days (Uptown's share of chain purchases), compared with the same weeks before the change. This controls for chain-wide growth and the Tuesday pattern.
* **Primary outcome:** Uptown gross profit per day. **Guardrail:** Uptown's share of chain purchases.
* **Review weekly; stop early** if Uptown's share of chain purchases falls by more than the break-even for the half-gap step (see table) for **two consecutive weeks**.
* **Decide at the end** only if the result is clear. Below we estimate how large a share change can be told apart from normal week-to-week noise. **Four weeks does not guarantee an answer.** If the result is inside the noise band, extend the test or keep prices as they are.

In [4]:
daily = raw.groupby(["Date", "Branch"])["Transaction ID"].nunique().unstack()
share = daily["Uptown"] / daily.sum(axis=1)
weekly = share.groupby(share.index.to_period("W")).mean().iloc[1:-1]   # drop partial first/last weeks
sd_w = weekly.diff().dropna().std() / np.sqrt(2)                       # week-to-week noise, trend removed by differencing
base = weekly.mean()
for weeks in [4, 8]:
    # minimum detectable change in mean weekly share (pre vs post, equal weeks, 80% power, 5% two-sided)
    mde = 2.8 * sd_w * np.sqrt(2 / weeks)
    print(f"{weeks} weeks before vs {weeks} after: detectable change in Uptown's share ≈ {mde:.4f} "
          f"(≈ {mde/base:.1%} of its {base:.1%} share) — Planning estimate")

4 weeks before vs 4 after: detectable change in Uptown's share ≈ 0.0139 (≈ 7.3% of its 19.0% share) — Planning estimate
8 weeks before vs 8 after: detectable change in Uptown's share ≈ 0.0098 (≈ 5.2% of its 19.0% share) — Planning estimate


This is a planning estimate under the assumption that weekly noise stays as in H1. With 4 weeks on each side, the detectable change (about 7%) is **about the same as the half-gap break-even (7.0%)**, so a 4-week test could miss a drop that just wipes out the gain. With 8 weeks it is about 5%. We therefore recommend 8 weeks if the board can accept it. Either way, the weekly guardrail, not the end-of-test result, is what protects the business from a large drop.

## 4. 10% discount: test removal, don't just remove it

**Facts.** The 10% discount appears on about 5.7% of lines at every branch, time and product. It cost $1,707 in H1. The data does not show whether it brings in sales.

In [5]:
d10 = raw[raw["Discount Rate"] == 0.10]
given = (d10.Quantity * d10["Unit Price (USD)"] * 0.10).sum()
print(f"Fact: 10% discount given in H1: ${given:,.0f} (${given/DAYS:.2f}/day chain-wide)")
print(f"Conditional annualised gross-profit opportunity: ${given*ANN:,.0f}/yr, only if removing it loses no sales")

gp_day = raw.groupby("Date")["Line Gross Profit (USD)"].sum()
X = pd.get_dummies(gp_day.index.day_name()).astype(float).values
X = np.column_stack([X, np.arange(len(gp_day))])
resid = gp_day.values - X @ np.linalg.lstsq(X, gp_day.values, rcond=None)[0]
sd = resid.std(ddof=X.shape[1])
for n in [14, 28, 56]:
    mde = 2.8 * sd * np.sqrt(2 / n)
    print(f"{n} discount days vs {n} no-discount days (randomly assigned within weeks): "
          f"detectable gross-profit change ≈ ${mde:,.0f}/day vs ${given/DAYS:.2f}/day at stake — Planning estimate")

Fact: 10% discount given in H1: $1,707 ($9.43/day chain-wide)
Conditional annualised gross-profit opportunity: $3,443/yr, only if removing it loses no sales
14 discount days vs 14 no-discount days (randomly assigned within weeks): detectable gross-profit change ≈ $98/day vs $9.43/day at stake — Planning estimate
28 discount days vs 28 no-discount days (randomly assigned within weeks): detectable gross-profit change ≈ $69/day vs $9.43/day at stake — Planning estimate
56 discount days vs 56 no-discount days (randomly assigned within weeks): detectable gross-profit change ≈ $49/day vs $9.43/day at stake — Planning estimate


**Reading.** The discount is worth about $9/day chain-wide. Normal day-to-day variation in gross profit is far larger, so even a two-month test **cannot reliably detect** an effect of that size. The test is therefore a **risk check**, not a measurement:

* **Design:** remove the 10% discount on randomly chosen days, balanced within each week, at all branches.
* **Stop rule (guardrail for clear harm):** at each weekly review, restore the discount if no-discount days are below discount days by more than the detectable amount for the days observed so far (≈ $69/day at 4 weeks), i.e. only when harm is clearly larger than noise. A stop rule set at the $9/day at stake would trigger on noise alone.
* **Review rule:** if no clear harm appears after 8 weeks, the board can keep it removed and keep monitoring. This is a judgement under uncertainty: the test cannot prove the $3.4k/yr is a real saving.

## 5. Board decision table

In [6]:
up = pricing[(pricing.branch == "Uptown")]
board = pd.DataFrame([
    {"rank": 1, "decision": "Audit Garden's cost table before any margin-based decision about Garden",
     "conditional annualised GP opportunity": f"up to ${gap*ANN:,.0f}/yr",
     "condition": "Garden really pays more than Harbour and equivalent terms are feasible",
     "evidence": "Fact: ÷0.7 rule on 45/45 products (Step 3)"},
    {"rank": 2, "decision": "Run an Uptown pricing experiment (half the gap to Harbour first)",
     "conditional annualised GP opportunity": f"${up.iloc[0]['opportunity if volume unchanged ($/yr)']:,}/yr (half gap) to ${up.iloc[1]['opportunity if volume unchanged ($/yr)']:,}/yr (full gap)",
     "condition": f"volume unchanged; positive while volume loss < {up.iloc[0]['break-even volume loss']} (half) / {up.iloc[1]['break-even volume loss']} (full)",
     "evidence": "Fact: prices ≈91.4% of Harbour's; demand response unknown"},
    {"rank": 3, "decision": "Test removing the 10% discount as a risk-checked experiment",
     "conditional annualised GP opportunity": f"up to ${given*ANN:,.0f}/yr",
     "condition": "removal loses no sales; a short test cannot confirm this",
     "evidence": "Fact: $1,707 in H1, no targeting pattern"},
    {"rank": 4, "decision": "Do not repeat Harbour's 15% campaign unchanged; test with randomised offer days",
     "conditional annualised GP opportunity": "avoids an estimated −$308 / −$391 GP per campaign",
     "condition": "model assumptions in Step 2",
     "evidence": "Model estimate (Step 2)"},
    {"rank": 5, "decision": "Validate customer-history targeting before committing marketing budget",
     "conditional annualised GP opportunity": "no dollar figure",
     "condition": "—",
     "evidence": "No demonstrated advantage over random selection (Step 4)"},
])
board

,rank,decision,conditional annualised GP opportunity,condition,evidence
0,1,Audit Garden's cost table before any margin-ba...,"up to $32,111/yr",Garden really pays more than Harbour and equiv...,Fact: ÷0.7 rule on 45/45 products (Step 3)
1,2,Run an Uptown pricing experiment (half the gap...,"$5,272/yr (half gap) to $10,545/yr (full gap)",volume unchanged; positive while volume loss <...,Fact: prices ≈91.4% of Harbour's; demand respo...
2,3,Test removing the 10% discount as a risk-check...,"up to $3,443/yr",removal loses no sales; a short test cannot co...,"Fact: $1,707 in H1, no targeting pattern"
3,4,Do not repeat Harbour's 15% campaign unchanged...,avoids an estimated −$308 / −$391 GP per campaign,model assumptions in Step 2,Model estimate (Step 2)
4,5,Validate customer-history targeting before com...,no dollar figure,—,No demonstrated advantage over random selectio...
